In [12]:
import selenium
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
from html import unescape
from bs4 import BeautifulSoup
import logging
import random

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Set up Chrome driver with options
chrome_options = Options()

chrome_options.add_argument("--disable-gpu")

chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36")
try:
    driver = webdriver.Chrome(options=chrome_options)
except Exception as e:
    logging.error(f"Erreur lors de l'initialisation du WebDriver: {e}")
    exit(1)
wait = WebDriverWait(driver, 40)

# Define the mapping lists
mapping_lists = {
    "Face": {
        "Primer": ["primer", "pore", "sunburst"],
        "Concealer": ["concealer"],
        "Foundation": ["foundation"],
        "Blush": ["blush", "festive", "cheeky", "use balm", "lip cream"],
        "Highlighter": ["highlighter", "highlighting", "glow stick", "shimmer", "eye pencil"],
        "Setting Spray": ["spray"],
        "Powder": ["powder"],
        "Palette": ["palette"],
        "Contour & Bronzer": ["bronzer", "contour", "tan", "sun", "pen-light", "dark", "nude", "sand", "bronzing"],
    },
    "Eyes": {
        "Eyeliner": ["eyeliner", "liner", "wand"],
        "Eyeshadow": ["eyeshadow", "tone shadow", "jelly", "glitter", "flake"],
        "Eyeshadow Palette": ["palette", "stack"],
        "Mascara": ["mascara"],
        "Eyebrow": ["brow", "eyebrow", "control hair"],
        "Lashes": ["lash", "lashes"],
        "Eye Primer": ["under eye"],
    },
    "Lips": {
        "Lipstick": ["lipstick", "Lip Set", "stacks"],
        "Lip Gloss": ["gloss", "Hydra Jelly", "care", "Mask", "oil", "Soft", "jam"],
        "Lip Liner": ["liner", "marker", "duo"],
        "Lip Balm": ["balm"],
        "Lip Kit": ["kit"],
        "Lip Plumper": ["lip plumper"],
        "Lip Tint": ["lip tint", "Cheek Tint"]
    },
    "Tools & Others": {
        "Brush Set": ["brush set"],
        "Brush": ["brush"],
        "Sponge": ["sponge"],
        "Puff": ["puff"],
        "Makeup Bag": ["bag"],
        "Mirror": ["mirror"],
        "Brush Cleaner": ["cleaner"],
        "Eyelash Tool": ["eyelash"],
        "Eyelash Curler": ["curler"],
        "Brows & Lash Tools": ["mermaid"],
        "Facial Cleaning Tools": ["mattifying"],
        "Eyelashes and Adhesives Kits": ["Magnetizing Eyes"]
    },
    "Hair Tools": {
        "Hair Straightener Iron": ["straightener"],
        "Hot-Air Brushes & Straight Hair Combs": ["sleek", "blowout", "straight-combs"],
        "Curling Tongs & Curling Wands": ["curler", "curling", "curl", "iron"],
    },
    "Hair Care": {
        "Hair Treatment": ["thermal", "glow", "leave-in", "oil"],
        "Hair Coloring": ["dye"],
        "Styling Products": ["Hold"]
    },
}

# Build the final keyword→subcategory map used in matching
subcategory_keywords = {
    cat: {
        kw.lower(): subcat
        for subcat, keywords in mapping_lists[cat].items()
        for kw in keywords
    }
    for cat in mapping_lists
}

data = []
best_seller_products = set()

def extract_image_urls_with_bs4(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    image_urls = []
    try:
        gallery = soup.find("div", class_="fun-pc-std-gallery-thumbnails")
        if gallery:
            img_tags = gallery.find_all("img", class_="fun-pc-image-img")
            for img in img_tags:
                src = img.get("src")
                if src:
                    if src.startswith("//"):
                        src = "https:" + src
                    image_urls.append(src)
        else:
            logging.warning("Galerie d'images non trouvée.")
    except Exception as e:
        logging.error(f"Erreur BeautifulSoup (images): {e}")
    return image_urls

def extract_ingredients_with_bs4(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    ingredients_text = "N/A"
    labels_list = []
    try:
        ingredients_block = soup.find("span", string="Ingredients")
        if ingredients_block:
            container = ingredients_block.find_parent("div", class_="fun-pc-std-collapse-wrapper")
            if container:
                label_divs = container.select(".style_label__chEXT")
                labels_list = [div.get_text(strip=True) for div in label_divs]
                ing_text_block = container.select_one(".fun-pc-ellipsis-wrapper div")
                if ing_text_block:
                    ingredients_text = ing_text_block.get_text(strip=True)
        else:
            logging.warning("Bloc ingrédients non trouvé.")
    except Exception as e:
        logging.error(f"Erreur BeautifulSoup (ingredients): {e}")
    return ingredients_text, labels_list

def click_main_view_more():
    while True:
        try:
            btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".fun-pc-std-list-showMore button")))
            driver.execute_script("arguments[0].click()", btn)
            time.sleep(random.uniform(2, 3))
        except:
            break

def get_product_details(el):
    d = {"Name": "N/A", "Price": "N/A", "Stars": 0.0}
    try:
        d["Name"] = unescape(el.find_element(By.CSS_SELECTOR, ".fun-pc-sg-goods-cell-name").text.strip())
    except:
        pass
    try:
        d["Price"] = el.find_element(By.CSS_SELECTOR, ".fun-pc-sg-goods-cell-salePrice").text.strip()
    except:
        pass
    try:
        stars = 0.0
        for star in el.find_elements(By.CSS_SELECTOR, ".fun-pc-std-rate-active"):
            pct = star.get_attribute("style").split(":")[1].replace("%", "").replace(";", "")
            stars += int(pct) / 100
        d["Stars"] = stars
    except:
        pass
    return d

def match_nlp_attributes(name, category):
    nm = name.lower()
    for kw, subcat in subcategory_keywords.get(category, {}).items():
        if kw in nm:
            return subcat
    return "N/A"

def get_product_reviews():
    reviews = []
    try:
        # Check if review section exists with a shorter timeout
        review_wait = WebDriverWait(driver, 10)  # Reduced timeout for reviews
        try:
            review_wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".fun-pc-sg-review-item-wrapper")))
        except:
            logging.info("Aucune section de reviews trouvée sur cette page.")
            return reviews  # Return empty list if no reviews

        # Scroll to ensure reviews are loaded
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(1, 2))  # Allow JavaScript to render

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        review_elements = soup.find_all('div', class_='fun-pc-sg-review-item-wrapper')
        if not review_elements:
            logging.info("Aucun élément de review trouvé via BeautifulSoup.")
            return reviews

        for review_element in review_elements:
            review = {}
            review['username'] = review_element.find('div', class_='fun-pc-sg-review-item-name').text.strip() if review_element.find('div', class_='fun-pc-sg-review-item-name') else "N/A"
            color_section = review_element.find('div', class_='fun-pc-sg-review-item-leftItem', string=lambda text: text and "Color:" in text)
            review['color'] = color_section.find_next('span').text.strip() if color_section else "N/A"
            size_section = review_element.find('div', class_='fun-pc-sg-review-item-leftItem', string=lambda text: text and "Size:" in text)
            review['size'] = size_section.find_next('span').text.strip() if size_section else "N/A"
            review['date'] = review_element.find('div', class_='fun-pc-sg-review-item-date').text.strip() if review_element.find('div', class_='fun-pc-sg-review-item-date') else "N/A"
            review['content'] = review_element.find('div', class_='fun-pc-sg-review-item-content').text.strip() if review_element.find('div', class_='fun-pc-sg-review-item-content') else "N/A"
            stars = len(review_element.find_all('span', class_='fun-pc-std-rate-active')) if review_element.find('span', class_='fun-pc-std-rate-active') else 0
            review['stars'] = stars
            reviews.append(review)
    except Exception as e:
        logging.error(f"Erreur lors de l'extraction des reviews: {e}")
        # Save page source for debugging
        with open(f"error_reviews_{int(time.time())}.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
    return reviews

def scrape_category(category_name):
    logging.info(f"Scraping category: {category_name}")
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".fun-pc-sg-goods-cell-wrapper")))
        click_main_view_more()
        products = driver.find_elements(By.CSS_SELECTOR, ".fun-pc-sg-goods-cell-wrapper")
        for idx, prod in enumerate(products, 1):
            try:
                driver.execute_script("arguments[0].scrollIntoView()", prod)
                time.sleep(0.1)
                details = get_product_details(prod)
                
                product_reviews = []
                image_urls = []
                ingredients_text = "N/A"
                labels_list = []
                original_window = driver.current_window_handle
                for attempt in range(2):  # Retry up to 2 times
                    try:
                        href = prod.find_element(By.TAG_NAME, "a").get_attribute("href")
                        logging.info(f"Tentative d'ouverture: {href}")
                        driver.execute_script("window.open(arguments[0])", href)
                        driver.switch_to.window(driver.window_handles[-1])
                        time.sleep(random.uniform(4, 6))  # Increased delay for page load
                        
                        product_reviews = get_product_reviews()
                        image_urls = extract_image_urls_with_bs4(driver)
                        if category_name not in ["Tools & Others", "Hair Tools"]:
                            ingredients_text, labels_list = extract_ingredients_with_bs4(driver)
                        
                        if category_name == "Hair Tools":
                            try:
                                plug_el = driver.find_element(
                                    By.XPATH,
                                    "//div[contains(@class,'style_mainAttrTitle__OV5Ab')]" +
                                    "/span[contains(text(),'Plug(Voltage):')]/following-sibling::span"
                                )
                                plug = plug_el.text.strip()
                                details["Name"] += f" [{plug}]"
                            except:
                                pass
                        break
                    except Exception as e:
                        logging.error(f"Erreur lors de l'ouverture de la page produit {details['Name']} (essai {attempt+1}): {e}")
                        if attempt == 1:
                            break
                        time.sleep(2)
                    finally:
                        try:
                            driver.close()
                            driver.switch_to.window(original_window)
                        except:
                            logging.warning("Impossible de fermer l'onglet produit.")
                        time.sleep(1)
                
                subcat = match_nlp_attributes(details["Name"], category_name)
                data.append({
                    "Category": category_name,
                    "Name": details["Name"],
                    "Price": details["Price"],
                    "Stars": round(details["Stars"], 1),
                    "Subcategory": subcat,
                    "Best Seller": 0,
                    "Reviews": product_reviews,
                    "Image URLs": image_urls,
                    "Ingredients": ingredients_text,
                    "Labels": labels_list
                })
            except Exception as e:
                logging.error(f"Erreur lors du traitement du produit {idx} dans {category_name}: {e}")
                continue
    except Exception as e:
        logging.error(f"Erreur dans scrape_category {category_name}: {e}")
        with open(f"error_page_{category_name}.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        raise

# Start from homepage with retry logic
for attempt in range(3):
    try:
        driver.get("https://www.sheglam.com/")
        time.sleep(random.uniform(3, 5))
        try:
            cookie_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".cookie-accept-button")))
            cookie_button.click()
        except:
            logging.info("Aucun pop-up de cookies trouvé.")
        break
    except Exception as e:
        logging.error(f"Échec du chargement de la page d'accueil (essai {attempt+1}): {e}")
        if attempt == 2:
            driver.quit()
            exit(1)
        time.sleep(5)

# Click through main categories
main_categories = {
    "Face": "//div[contains(@class,'fun-pc-sheglam-menu-item')]//span[contains(text(),'Face')]",
    "Eyes": "//div[contains(@class,'fun-pc-sheglam-menu-item')]//span[contains(text(),'Eyes')]",
    "Lips": "//div[contains(@class,'fun-pc-sheglam-menu-item')]//span[contains(text(),'Lips')]",
    "Tools & Others": "//div[contains(@class,'fun-pc-sheglam-menu-item')]//span[contains(text(),'Tools')]"
}

for category_name, xpath in main_categories.items():
    try:
        logging.info(f"Tentative de scraping pour {category_name}")
        wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class,'fun-pc-sheglam-menu-item')]")))
        category_btn = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].scrollIntoView(true);", category_btn)
        driver.execute_script("arguments[0].click()", category_btn)
        time.sleep(random.uniform(2, 3))
        scrape_category(category_name)
        driver.back()
        time.sleep(random.uniform(2, 3))
        df = pd.DataFrame(data)
        df.to_csv(f"sheglam_{category_name}.csv", index=False)
    except Exception as e:
        logging.error(f"Échec du scraping de {category_name}: {e}")
        continue

# Click on SHEGLAM HAIR tab
try:
    hair_tab = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//div[contains(@class,'fun-pc-sg-home-tab-tabItem') and contains(text(),'SHEGLAM HAIR')]")
    ))
    driver.execute_script("arguments[0].click()", hair_tab)
    time.sleep(2)
    
    # Click through hair subcategories
    hair_subcategories = {
        "Hair Tools": "//span[contains(@class,'fun-pc-sg-bcc-menu-menuTitle') and contains(text(),'Hot Tools')]",
        "Hair Care": "//span[contains(@class,'fun-pc-sg-bcc-menu-menuTitle') and contains(text(),'Hair Care')]"
    }
    
    for subcat_name, xpath in hair_subcategories.items():
        try:
            subcat_btn = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
            driver.execute_script("arguments[0].click()", subcat_btn)
            time.sleep(2)
            scrape_category(subcat_name)
            driver.back()
            time.sleep(2)
        except Exception as e:
            print(f"Failed to scrape {subcat_name}: {str(e)}")
            df = pd.DataFrame(data)
        df.to_csv("sheglam_hair.csv", index=False)
except Exception as e:
    print(f"Failed to navigate to SHEGLAM HAIR: {str(e)}")
    



2025-05-13 09:47:35,925 - INFO - Aucun pop-up de cookies trouvé.
2025-05-13 09:47:39,028 - ERROR - Échec du scraping de Hair Tools : Message: invalid selector: Unable to locate an element with the xpath expression //span Fcontient(@class,'fun-pc-sg-bcc-menu-menuTitle') et contient(texte(),'Hot Tools')] because of the following error:
SyntaxError: Failed to execute 'evaluate' on 'Document': The string '//span Fcontient(@class,'fun-pc-sg-bcc-menu-menuTitle') et contient(texte(),'Hot Tools')]' is not a valid XPath expression.
  (Session info: chrome=136.0.7103.93); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalid-selector-exception
Stacktrace:
	GetHandleVerifier [0x00007FF6D70BCF25+75717]
	GetHandleVerifier [0x00007FF6D70BCF80+75808]
	(No symbol) [0x00007FF6D6E88F9A]
	(No symbol) [0x00007FF6D6E90998]
	(No symbol) [0x00007FF6D6E93A0C]
	(No symbol) [0x00007FF6D6E93ADF]
	(No symbol) [0x00007FF6D6EDED77]
	(No symbol

In [22]:
# Scrape best sellers
try:
    driver.get("https://www.sheglam.com/collection/Best-Sellers-30003?sort=most_popular")
    time.sleep(random.uniform(2, 3))
    click_main_view_more()
    
    for prod in driver.find_elements(By.CSS_SELECTOR, ".fun-pc-sg-goods-cell-wrapper") :
        try :
            nm = unescape(prod.find_element(By.CSS_SELECTOR, ".fun-pc-sg-goods-cell-name").text.strip())
            best_seller_products.add(nm)
        except :
            passer
except Exception as e :
    logging.error(f"Échec du scraping des best sellers : {e}")

# Update best-seller flag
for item in data :
    if item["Name"] in best_seller_products :
        item["Best Seller"] = 1

# Save final CSV
try :
    df = pd.DataFrame(data)
    df = df[["Category", "Name", "Price", "Stars", "Subcategory", "Best Seller", "Image URLs", "Ingredients", "Labels"]]
    df.to_csv("sheglam_products.csv", index=False)
except Exception as e :
    logging.error(f"Erreur lors de l'enregistrement du CSV : {e}")

driver.quit()
logging.info("Scraping completed!")

[{'Category': 'Hair Tools',
  'Name': 'It-Curl One-Touch Instant Curler - 32mm []',
  'Price': 'MX$981.92',
  'Stars': 5.0,
  'Subcategory': 'Curling Tongs & Curling Wands',
  'Best Seller': 0,
  'Reviews': [{'username': 'G***7',
    'color': 'N/A',
    'size': 'N/A',
    'date': '2025.02.17',
    'content': 'J’adore cet appareil.',
    'stars': 5},
   {'username': 'G***5',
    'color': 'N/A',
    'size': 'N/A',
    'date': '2025.02.09',
    'content': 'Best curler ever. Can’t live without it.',
    'stars': 5},
   {'username': 'C***2',
    'color': 'N/A',
    'size': 'N/A',
    'date': '2025.01.28',
    'content': 'I declare this my favorite.',
    'stars': 5},
   {'username': 'P***1',
    'color': 'N/A',
    'size': 'N/A',
    'date': '2025.01.24',
    'content': 'Super quick. 🌟',
    'stars': 5},
   {'username': 'O***3',
    'color': 'N/A',
    'size': 'N/A',
    'date': '2025.01.22',
    'content': 'Works so fast. Done in 10 minutes. ✨',
    'stars': 5}],
  'Image URLs': ['https://